In [17]:
import torch
import json
import pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import os

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [5]:
model_name = "sarvamai/sarvam-m"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)

quantization_config = BitsAndBytesConfig(load_in_8bit=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    quantization_config=quantization_config,
    device_map="auto"
)

Loading checkpoint shards:   0%|          | 0/10 [00:00<?, ?it/s]

/home/mayank/tableQA/telemanas_project_env/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.4` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/home/mayank/tableQA/telemanas_project_env/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.4` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


In [6]:
model.device

device(type='cuda', index=0)

In [7]:
# prepare the model input
prompt = "What is AI in one sentence."

messages = [{"role": "user", "content": prompt}]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    enable_thinking=True,  # Switches between thinking and non-thinking modes. Default is True.
)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(**model_inputs, max_new_tokens=8192)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]) :].tolist()
output_text = tokenizer.decode(output_ids)

if "</think>" in output_text:
    reasoning_content = output_text.split("</think>")[0].rstrip("\n")
    content = output_text.split("</think>")[-1].lstrip("\n").rstrip("</s>")
else:
    reasoning_content = ""
    content = output_text.rstrip("</s>")

print("reasoning content:", reasoning_content)
print("content:", content)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


/home/mayank/tableQA/telemanas_project_env/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:315: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


reasoning content: Okay, the user is asking, "What is AI in one sentence." Let me start by understanding the context. They probably want a concise definition without too much jargon. Since they mentioned "in one sentence," brevity is key.

First, I need to define AI clearly. AI stands for Artificial Intelligence. It's about machines or systems that can perform tasks requiring human-like intelligence. But I should make it simple. Maybe mention things like learning, reasoning, problem-solving. Also, include examples like recognizing speech or making decisions. But keep it all in one sentence.

Wait, the user might be a student or someone new to the topic. They might need a basic understanding. So avoid technical terms. Use everyday examples. Maybe start with "AI is technology that enables machines to..." Then list the capabilities. Make sure it's comprehensive but not overwhelming.

Also, considering the user's possible follow-up questions. If they get a good one-sentence answer, they mi

In [11]:
# Few-shot template 
FEW_SHOT_HINDI = """
# Instruction:
You are a helpful assistant who generates answers from a Hindi table to answer Hindi questions.  
Use the below example to guide the format. 

## Example:

### Input:
 ले ट्रुनिन ने किन फिल्मों में भूमिका निभाई थी?  

<column>  वर्ष | शीर्षक | भूमिका  
<row 1> 2014 | See No Evil 2 | जैकब गुडनाइट  
<row 2> 2016 | Countdown | ले ट्रुनिन  
<row 3> 2017 | Meltdown | ले ट्रुनिन  

### Response (complete this):
<column> शीर्षक  
<row 1> Countdown  
<row 2> Meltdown  

now answer the following question, generate only the ouput table and nothing else.
"""

FEW_SHOT_TELUGU = """
# Instruction:
You are a helpful assistant who generates answers from a Telugu table to answer Telugu questions.  
Use the below example to guide the format. 

## Example:

### Input:
 లే ట్రునిన్ ఏ సినిమాలలో పాత్ర పోషించాడు?

<column>  సంవత్సరం | శీర్షిక | పాత్ర  
<row 1> 2014 | See No Evil 2 | జేకబ్ గుడ్‌నైట్  
<row 2> 2016 | Countdown | లే ట్రునిన్  
<row 3> 2017 | Meltdown | లే ట్రునిన్  

### Response (complete this):
<column> శీర్షిక  
<row 1> Countdown  
<row 2> Meltdown  

now answer the following question, generate only the ouput table and nothing else.
"""

FEW_SHOT_BENGALI = """
# Instruction:
You are a helpful assistant who generates answers from a Bengali table to answer Bengali questions.  
Use the below example to guide the format. 

## Example:

### Input:
 লে ট্রুনিন কোন কোন সিনেমায় অভিনয় করেছেন?

<column>  বছর | শিরোনাম | চরিত্র  
<row 1> 2014 | See No Evil 2 | জ্যাকব গুডনাইট  
<row 2> 2016 | Countdown | লে ট্রুনিন  
<row 3> 2017 | Meltdown | লে ট্রুনিন  

### Response (complete this):
<column> শিরোনাম  
<row 1> Countdown  
<row 2> Meltdown  

now answer the following question, generate only the ouput table and nothing else.
"""


In [12]:
# Load test.json
with open("test.json", "r") as f:
    test_data = json.load(f)

# Few-shot template (Hindi, as per your example)
FEW_SHOT = """
# Instruction:
You are a helpful assistant who generates answers from a Hindi table to answer Hindi questions.  
Use the below example to guide the format. 

## Example:

### Input:
 ले ट्रुनिन ने किन फिल्मों में भूमिका निभाई थी?  

<column>  वर्ष | शीर्षक | भूमिका  
<row 1> 2014 | See No Evil 2 | जैकब गुडनाइट  
<row 2> 2016 | Countdown | ले ट्रुनिन  
<row 3> 2017 | Meltdown | ले ट्रुनिन  

### Response (complete this):
<column> शीर्षक  
<row 1> Countdown  
<row 2> Meltdown  

now answer the following question, generate only the ouput table and nothing else.
"""

def table_to_text(table):
    if not table:
        return ""
    columns = list(table[0].keys())
    header = " | ".join(columns)
    rows = []
    for i, row in enumerate(table):
        row_str = " | ".join(str(row[col]) for col in columns)
        rows.append(f"<row {i+1}> {row_str}")
    return f"<column>\n{header}\n" + "\n".join(rows)

def resultdf_to_text(result_df):
    if not result_df:
        return ""
    df = pd.DataFrame(result_df)
    return df.to_string(index=False)



In [13]:
# total_queries = sum(len(table["queries"]) for table in test_data)

# with tqdm(total=total_queries, desc="Processing queries") as pbar:
#     for table in test_data:
#         table_name = table["table_name"]
#         full_table = table["full_table_df"]
#         table_text = table_to_text(full_table)
#         for q in table["queries"]:
#             question = q["question"]
#             result_df = q.get("result_df", [])
#             # Compose the prompt
#             prompt = FEW_SHOT + f"\n###Input:\n{question}\n{table_text}\n###Response:\n"
#             # Prepare chat template for Sarvam
#             messages = [{"role": "user", "content": prompt}]
#             text = tokenizer.apply_chat_template(
#                 messages,
#                 tokenize=False,
#                 enable_thinking=True,
#             )
#             print("="*80)
#             print(f"Question: {question}")
#             print(f"Table: {table_name}")
#             model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
#             generated_ids = model.generate(**model_inputs, max_new_tokens=1024)
#             output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
#             output_text = tokenizer.decode(output_ids)
#             # Extract only the content after ###Response:
#             if "###Response:" in output_text:
#                 model_response = output_text.split("###Response:")[-1].strip()
#             else:
#                 model_response = output_text.strip()
            
#             print("\nModel Response:\n", model_response)
#             print("\nActual Result DF:\n", resultdf_to_text(result_df))
#             print("="*80)
#             pbar.update(1)
            
        

In [14]:
def run_tableqa_testset(model, tokenizer, model_instruction, testset_path, result_path, sanity=False, thinking_mode=True):
    """
    Loads a test set from testset_path, prompts the model for each query, and writes results to result_path.
    If sanity=True, processes only one question and saves with sanity_ prefix.
    """
    generation_config = {
        "temperature": 0.6,
        "top_p": 0.95,
        "top_k": 20,
        "min_p": 0.0,
        # "do_sample": True,
        "max_new_tokens": 1024
    }
    with open(testset_path, "r", encoding="utf-8") as f:
        test_data = json.load(f)

    tables_to_process = test_data if not sanity else [test_data[0]]
    total_queries = 1 if sanity else sum(len(table["queries"]) for table in tables_to_process)

    with tqdm(total=total_queries, desc="Processing queries") as pbar:
        for table in tables_to_process:
            full_table = table["full_table_df"]
            table_text = table_to_text(full_table)
            queries_to_process = [table["queries"][0]] if sanity else table["queries"]
            
            for q in queries_to_process:
                question = q["question"]
                prompt = f"\n###Input:\n{question}\n{table_text}\n###Response:\n"
                messages = [
                    {"role": "system", "content": model_instruction},
                    {"role": "user", "content": prompt}
                ]
                text = tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True,
                    enable_thinking=thinking_mode # Switches between thinking and non-thinking modes. Default is True.
                )
                model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
                generated_ids = model.generate(**model_inputs, **generation_config)
                output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
                output_text = tokenizer.decode(output_ids)

                # Try to split into thinking content and response content
                try:
                    # Look for </think> token
                    index = len(output_ids) - output_ids[::-1].index(151668)  # 151668 is </think> token
                    thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
                    content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")
                except ValueError:
                    # If </think> not found, put everything in content
                    thinking_content = ""
                    content = output_text.strip()

                # Extract response after ###Response: if present
                if "###Response:" in content:
                    content = content.split("###Response:")[-1].strip()

                q["model_response"] = {
                    "thinking_content": thinking_content,
                    "content": content
                }

                pbar.update(1)
                if sanity:
                    print("prompt:\n", prompt)
                    print("Thinking content:", thinking_content)
                    print("Response content:", content)
                    
                    # Save with sanity_ prefix for sanity check
                    dir_path = os.path.dirname(result_path)
                    base_name = os.path.basename(result_path)
                    sanity_path = os.path.join(dir_path, "sanity_" + base_name)
                    with open(sanity_path, "w", encoding="utf-8") as f:
                        json.dump([table], f, ensure_ascii=False, indent=2)
                    return

    if not sanity:
        with open(result_path, "w", encoding="utf-8") as f:
            json.dump(test_data, f, ensure_ascii=False, indent=2)

In [19]:
run_tableqa_testset(
    model=model,
    tokenizer=tokenizer, 
    model_instruction=FEW_SHOT_HINDI, 
    testset_path="data/hindi/hindi_testset.json", 
    result_path="data/hindi/sarvam_results_nothink.json", 
    sanity=True,
    thinking_mode=False
)

Processing queries: 100%|██████████| 1/1 [00:10<00:00, 10.37s/it]

prompt:
 
###Input:
उन देशों के ग्रीनहाउस गैस उत्सर्जन में बदलाव (1990-2004) LULUCF को छोड़कर प्रतिशत में क्या है जिनकी संधि बाध्यता 2008-2012 के लिए 11.0 है?
<column>
देश*: | ग्रीनहाउस गैस उत्सर्जन में बदलाव.(1990-2004)LULUCF को छोड़कर% | ग्रीनहाउस गैस उत्सर्जन में बदलाव.(1990-2004)LULUCF सहित% | संधि बाध्यता 2008-2012
<row 1> डेनमार्क | -19.0 | -22.2 | 11.0
<row 2> जर्मनी, | -17.0 | -18.2 | 8%
<row 3> कनाडा | 27.0 | 26.6 | 6.0
<row 4> (ऑस्ट्रेलिया) | 25.0 | 5.2 | 8%
<row 5> स्पेन | 49.0 | 50.4 | 8%
<row 6> नॉर्वे | 10.0 | -18.7 | 1%
<row 7> न्यूजीलैंड | 21.0 | 17.9 | 0.0
<row 8> फ़्रांस | -0.8 | -6.1 | 8%
<row 9> ग्रीस | 27.0 | 25.3 | 8%
<row 10> आयरलैण्ड | 23.0 | 22.7 | 8%
<row 11> जापान, | 6.5 | 5.2 | 6.0
<row 12> यूनाइटेड किंगडम | -14.0 | -58.8 | 8%
<row 13> पुर्तगाल | 41.0 | 28.9 | 8%
<row 14> यूरोपीय संघ-15 | -0.8 | -2.6 | 8%
###Response:

Thinking content: 
Response content: <column> देश*  
<row 1> डेनमार्क</s>
